# Ejercicios: tasa-distorsión

La función de tasa-distorsión $R(D)$ describe la mínima tasa de compresión que permite
una distorsión media $\leq D$. En este cuaderno calcularás $R(D)$ para dos fuentes
canónicas y verificarás las propiedades teóricas.

**Artículo asociado:** [Teoría de la tasa-distorsión](../../02-teoria-informacion/09-teoria-tasa-distorsion.md)

In [ ]:
import math

def log2(x):
    return math.log2(x) if x > 0 else 0.0

def h_bin(p):
    """Entropía binaria H_b(p)."""
    if p <= 0 or p >= 1:
        return 0.0
    return -p * log2(p) - (1-p) * log2(1-p)

print('Utilidades cargadas.')

## Ejercicio 1: R(D) para fuente Bernoulli con distorsión Hamming

Para $X \sim \text{Ber}(1/2)$ con distorsión Hamming $d(x,\hat{x}) = \mathbf{1}[x \neq \hat{x}]$:

$$R(D) = 1 - H_b(D), \quad 0 \leq D \leq 1/2$$

donde $H_b(D) = -D\log_2 D - (1-D)\log_2(1-D)$ es la entropía binaria.

**Tarea:** implementa `R_bernoulli(D)` y traza la curva $R(D)$.

In [ ]:
def R_bernoulli(D):
    """
    Función de tasa-distorsión para Ber(1/2) con distorsión Hamming.
    R(D) = 1 - H_b(D) para 0 <= D <= 0.5; R(D) = 0 para D > 0.5.
    """
    # TODO: implementar
    pass

# Descomenta cuando hayas implementado R_bernoulli:
# print('R(D) para Bernoulli(1/2) con distorsión Hamming:')
# print(f'{"D":>8}  {"R(D) calculada":>16}  {"R(D) manual":>14}')
# casos = [(0.0, 1.0), (0.1, None), (0.2, None), (0.3, None), (0.5, 0.0)]
# for D, manual in casos:
#     Rd = R_bernoulli(D)
#     print(f'{D:>8.2f}  {Rd:>16.4f}  {str(manual):>14}')

### Solución de referencia para el ejercicio 1

In [ ]:
def R_bernoulli_ref(D):
    if D <= 0:
        return 1.0
    if D >= 0.5:
        return 0.0
    return 1.0 - h_bin(D)

print('Curva R(D) para Bernoulli(1/2) con distorsión Hamming:')
print(f'{"D":>8}  {"R(D)":>10}  {"Interpretación":>40}')
print('-' * 65)
casos = [
    (0.00, 'sin pérdida → entropía completa (1 bit)'),
    (0.05, '5% error → 0.714 bits/símbolo'),
    (0.10, '10% error'),
    (0.20, '20% error'),
    (0.30, '30% error'),
    (0.40, '40% error'),
    (0.50, '50% error → adivinar al azar: 0 bits necesarios'),
]
for D, desc in casos:
    Rd = R_bernoulli_ref(D)
    print(f'{D:>8.2f}  {Rd:>10.4f}  {desc}')

print()
print('Verificación de propiedades:')
print(f'  R(0) = {R_bernoulli_ref(0):.4f}  (debe ser H(X) = 1 bit)')
print(f'  R(0.5) = {R_bernoulli_ref(0.5):.4f}  (debe ser 0)')
# R(D) debe ser decreciente y convexa
Ds = [i * 0.05 for i in range(11)]
Rs = [R_bernoulli_ref(D) for D in Ds]
decreciente = all(Rs[i] >= Rs[i+1] for i in range(len(Rs)-1))
print(f'  Decreciente: {decreciente}')

## Ejercicio 2: R(D) para fuente Gaussiana con MSE

Para $X \sim \mathcal{N}(0, \sigma^2)$ con distorsión MSE $d(x,\hat{x}) = (x-\hat{x})^2$:

$$R(D) = \frac{1}{2}\log_2\frac{\sigma^2}{D}, \quad 0 \leq D \leq \sigma^2$$

**Tarea:** implementa `R_gaussiana(D, sigma2)` y verifica propiedades.

In [ ]:
def R_gaussiana(D, sigma2):
    """
    Función de tasa-distorsión para N(0, sigma^2) con MSE.
    R(D) = 0.5 * log2(sigma^2/D) para 0 < D <= sigma^2.
    R(D) = 0 para D >= sigma^2.
    """
    # TODO: implementar
    pass

# Descomenta cuando hayas implementado R_gaussiana:
# sigma2 = 1.0
# print(f'R(D) para N(0,{sigma2}) con MSE:')
# for D in [0.01, 0.1, 0.25, 0.5, 0.75, 1.0]:
#     print(f'  D={D:.2f}  R(D)={R_gaussiana(D, sigma2):.4f}')

### Solución de referencia para el ejercicio 2

In [ ]:
def R_gaussiana_ref(D, sigma2):
    if D <= 0:
        return float('inf')
    if D >= sigma2:
        return 0.0
    return 0.5 * log2(sigma2 / D)

sigma2 = 1.0
print(f'R(D) para N(0,{sigma2}) con MSE:')
print(f'{"D":>8}  {"R(D)":>10}  {"Interpretación"}')
print('-' * 60)
casos_g = [
    (0.001, 'D muy pequeño → R grande'),
    (0.01,  'PSNR ≈ 20 dB'),
    (0.10,  '10% de la varianza'),
    (0.25,  '25% de la varianza → 1 bit'),
    (0.50,  '50% de la varianza → 0.5 bits'),
    (1.00,  'D = sigma^2 → R = 0 (ignorar la fuente)'),
]
for D, desc in casos_g:
    Rd = R_gaussiana_ref(D, sigma2)
    print(f'{D:>8.3f}  {Rd:>10.4f}  {desc}')

print()
print('Propiedad: cada bit adicional de tasa divide D por 4 (mejora 6 dB en PSNR):')
for R_target in [0.5, 1.0, 2.0, 3.0]:
    D_at_R = sigma2 / (2 ** (2 * R_target))
    print(f'  R = {R_target:.1f} bits → D = σ²/4^R = {sigma2}/{4**R_target:.0f} = {D_at_R:.4f}')

## Ejercicio 3: comparación de distribuciones

Compara $R(D)$ para distintas varianzas Gaussianas y para la Bernoulli.
¿Qué distribución necesita más bits para la misma calidad relativa de reconstrucción?

In [ ]:
print('Comparación de R(D) para distintas fuentes:')
print('(D expresado como fracción de la distorsión máxima)')
print()
print(f'{"D/D_max":>10}  {"Bernoulli(1/2)":>16}  {"Gauss(σ²=1)":>13}  {"Gauss(σ²=4)":>13}')
print('-' * 58)
fracciones = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
for f in fracciones:
    D_bern = f * 0.5      # D_max para Bernoulli es 0.5
    D_g1   = f * 1.0      # D_max para Gaussiana(sigma2=1) es 1.0
    D_g4   = f * 4.0      # D_max para Gaussiana(sigma2=4) es 4.0
    Rb  = R_bernoulli_ref(D_bern)
    Rg1 = R_gaussiana_ref(D_g1, 1.0)
    Rg4 = R_gaussiana_ref(D_g4, 4.0)
    print(f'{f:>10.2f}  {Rb:>16.4f}  {Rg1:>13.4f}  {Rg4:>13.4f}')

print()
print('Nota: Bernoulli y Gaussiana tienen formas muy distintas de R(D).')
print('Para la Gaussiana, R(D) → ∞ cuando D → 0 (imposible reconstrucción perfecta).')
print('Para la Bernoulli, R(0) = 1 bit (la fuente es finita).')

## Ejercicio 4: separación fuente-canal

El teorema de separación de Shannon establece que para transmitir una fuente Gaussiana de
varianza $\sigma^2$ a través de un canal Gaussiano de capacidad $C$, la distorsión mínima es:

$$D^* = \sigma^2 \cdot 2^{-2C}$$

Calcula $D^*$ para distintas SNR del canal.

In [ ]:
sigma2 = 1.0  # varianza de la fuente

def capacidad_gaussiana(SNR):
    return 0.5 * log2(1 + SNR)

def distorsion_optima(sigma2, SNR):
    """D* = sigma^2 * 2^(-2C) para canal Gaussiano con SNR dado."""
    C = capacidad_gaussiana(SNR)
    return sigma2 * (2 ** (-2 * C))

print('Separación fuente-canal para fuente N(0,1) y canal Gaussiano:')
print(f'{"SNR (dB)":>10}  {"SNR":>8}  {"Capacidad C":>14}  {"Distorsión D*":>15}')
print('-' * 55)
for snr_db in [0, 3, 6, 10, 13, 20, 30]:
    SNR = 10 ** (snr_db / 10)
    C = capacidad_gaussiana(SNR)
    D_star = distorsion_optima(sigma2, SNR)
    print(f'{snr_db:>10}  {SNR:>8.2f}  {C:>14.4f}  {D_star:>15.6f}')

print()
print('Con cada 3 dB de SNR, la capacidad crece ~0.5 bit y D* se divide aproximadamente por 2.')

## Reflexión

- $R(D)$ es siempre decreciente y convexa: a mayor distorsión tolerada, menos bits necesarios.
- Para la Bernoulli: $R(0) = H(X)$ (sin pérdida = entropía); $R(0.5) = 0$ (adivinar al azar).
- Para la Gaussiana: $R(D) = \infty$ cuando $D \to 0$ (continua = imposible reconstrucción exacta).
- Cada bit extra de tasa mejora la distorsión Gaussiana en 6 dB (divide D por 4).
- El teorema de separación fuente-canal garantiza que comprimir hasta $R(D)$ y transmitir
  a través de un canal de capacidad $C \geq R(D)$ es óptimo.

**Pregunta:** ¿por qué en la práctica los codecs de audio/video no alcanzan exactamente $R(D)$?
¿Qué limitaciones prácticas hay?